In [ ]:
#####     一位志愿者bpm，频率柱状图和一位志愿者心率检测报告
from scipy import stats
from sklearn.ensemble import IsolationForest
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # 使用非交互式后端


plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False  # 用来正常显示负号


class HealthWarningModel:
    def __init__(self, data_path):
        """
        初始化健康预警模型

        参数:
        - data_path: CSV数据文件路径
        """
        # 加载数据
        self.data = pd.read_csv(data_path)

        # 心率相关参数
        self.heart_rate_column = 'heart_rate'

        # 预警阈值设置（修复元组定义）
        self.warning_levels = {
            'green': (50, 100),   # 正常范围
            'yellow': [(40, 50), (100, 120)],  # 轻微异常
            'orange': [(30, 40), (120, 140)],  # 中度异常
            'red': [(0, 30), (140, 300)]  # 高风险
        }

    def calculate_statistical_thresholds(self):
        """
        计算心率的统计阈值

        返回:
        - 心率的统计阈值（均值、标准差）
        """
        heart_rates = self.data[self.heart_rate_column]
        mean = np.mean(heart_rates)
        std = np.std(heart_rates)

        return mean, std

    def detect_anomalies_isolation_forest(self, contamination=0.1):
        """
        使用孤立森林算法检测心率异常

        参数:
        - contamination: 预期异常数据比例

        返回:
        - 异常数据点
        """
        # 选择心率特征
        X = self.data[[self.heart_rate_column]]

        # 初始化并拟合孤立森林模型
        iso_forest = IsolationForest(
            contamination=contamination,
            random_state=42
        )

        # 预测异常
        anomaly_labels = iso_forest.fit_predict(X)

        return self.data[anomaly_labels == -1]

    def generate_warning_level(self, heart_rate):
        """
        根据心率生成预警级别

        参数:
        - heart_rate: 当前心率

        返回:
        - 预警级别（绿/黄/橙/红）
        """
        # 正常范围
        if self.warning_levels['green'][0] <= heart_rate <= self.warning_levels['green'][1]:
            return 'green'

        # 黄色预警
        for low, high in self.warning_levels['yellow']:
            if low <= heart_rate < high:
                return 'yellow'

        # 橙色预警
        for low, high in self.warning_levels['orange']:
            if low <= heart_rate < high:
                return 'orange'

        # 红色预警
        return 'red'

    def real_time_monitoring(self, current_heart_rate):
        """
        实时心率监测与预警

        参数:
        - current_heart_rate: 当前心率

        返回:
        - 预警信息字典
        """
        # 计算统计阈值
        mean, std = self.calculate_statistical_thresholds()

        # 检测异常
        anomalies = self.detect_anomalies_isolation_forest()

        # 生成预警级别
        warning_level = self.generate_warning_level(current_heart_rate)

        # 构建预警信息
        warning_info = {
            'current_heart_rate': current_heart_rate,
            'mean_heart_rate': mean,
            'heart_rate_std': std,
            'warning_level': warning_level,
            'anomalies_detected': len(anomalies),
            'potential_risks': {
                'atrial_fibrillation': warning_level in ['orange', 'red'],
                'high_blood_sugar_risk': warning_level == 'red'
            }
        }

        return warning_info

    def visualize_heart_rate_distribution(self):
        """
        可视化心率分布并保存图像，同时输出详细统计信息
        """
        heart_rates = self.data[self.heart_rate_column]

        # 创建图形
        plt.figure(figsize=(12, 8))

        # 绘制直方图
        n, bins, patches = plt.hist(
            heart_rates, bins=30, edgecolor='black', alpha=0.7)

        # 添加正态分布拟合曲线
        mu, std = np.mean(heart_rates), np.std(heart_rates)
        x = np.linspace(mu - 3*std, mu + 3*std, 100)
        p = stats.norm.pdf(x, mu, std)
        plt.plot(x, p * len(heart_rates) *
                 (bins[1] - bins[0]), 'r-', linewidth=2, label='正态分布拟合')

        # 设置标题和标签
        plt.title('心率分布统计分析', fontsize=15)
        plt.xlabel('心率', fontsize=12)
        plt.ylabel('频率', fontsize=12)

        # 添加网格
        plt.grid(axis='y', linestyle='--', alpha=0.7)

        # 添加图例
        plt.legend()

        # 标注统计信息
        plt.text(0.05, 0.95,
                 f'平均心率: {mu:.2f}\n'
                 f'标准差: {std:.2f}\n'
                 f'最小值: {heart_rates.min():.2f}\n'
                 f'最大值: {heart_rates.max():.2f}',
                 transform=plt.gca().transAxes,
                 verticalalignment='top',
                 bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

        # 保存图像
        plt.tight_layout()
        plt.savefig('heart_rate_distribution.png', dpi=300)
        plt.close()  # 关闭图形以释放内存

        # 打印详细统计信息
        print("心率统计信息:")
        print(f"平均心率: {mu:.2f}")
        print(f"标准差: {std:.2f}")
        print(f"最小值: {heart_rates.min():.2f}")
        print(f"最大值: {heart_rates.max():.2f}")
        print(f"中位数: {np.median(heart_rates):.2f}")

        # 返回统计信息字典，方便后续使用
        return {
            'mean': mu,
            'std': std,
            'min': heart_rates.min(),
            'max': heart_rates.max(),
            'median': np.median(heart_rates)
        }

    def analyze_user_heart_rate(self, user_id):
        """
        分析特定用户的心率数据

        参数:
        - user_id: 用户ID

        返回:
        - 用户心率详细分析报告
        """
        # 筛选特定用户的心率数据
        user_heart_rates = self.data[self.data['user_id']
                                     == user_id][self.heart_rate_column]

        # 如果没有找到用户数据
        if len(user_heart_rates) == 0:
            print(f"未找到用户 {user_id} 的心率数据")
            return None

        # 计算统计指标
        stats = {
            'user_id': user_id,
            'total_records': len(user_heart_rates),
            'min_heart_rate': user_heart_rates.min(),
            'max_heart_rate': user_heart_rates.max(),
            'mean_heart_rate': user_heart_rates.mean(),
            'median_heart_rate': user_heart_rates.median(),
            'std_heart_rate': user_heart_rates.std()
        }

        # 异常检测
        anomalies = self.detect_anomalies_isolation_forest(
            contamination=0.05  # 调整异常检测的敏感度
        )
        user_anomalies = anomalies[anomalies['user_id'] == user_id]
        stats['anomalies_count'] = len(user_anomalies)

        # 预警级别统计
        warning_levels = user_heart_rates.apply(self.generate_warning_level)
        warning_level_counts = warning_levels.value_counts()
        stats['warning_level_distribution'] = warning_level_counts.to_dict()

        # 高风险心率记录
        high_risk_records = user_heart_rates[warning_levels == 'red']
        stats['high_risk_records'] = list(high_risk_records)

        # 打印详细的心率分析信息
        print("\n用户心率分析详情：")
        print(f"用户ID: {user_id}")
        print(f"总记录数: {stats['total_records']}")
        print(f"最小心率: {stats['min_heart_rate']:.2f}")
        print(f"最大心率: {stats['max_heart_rate']:.2f}")
        print(f"平均心率: {stats['mean_heart_rate']:.2f}")
        print(f"心率中位数: {stats['median_heart_rate']:.2f}")
        print(f"心率标准差: {stats['std_heart_rate']:.2f}")
        print("\n心率预警级别分布：")
        for level, count in stats['warning_level_distribution'].items():
            print(f"{level}级预警: {count} 次")
        print(f"\n异常心率记录数: {stats['anomalies_count']}")

        if stats['high_risk_records']:
            print("\n高风险心率记录：")
            print(stats['high_risk_records'])

        return stats


# 使用示例
if __name__ == '__main__':
    # 初始化模型
    model = HealthWarningModel('cleaned_sleep_data_fixed_time.csv')

    # 分析特定用户心率数据
    user_id = 1066528
    user_analysis = model.analyze_user_heart_rate(user_id)

    if user_analysis:
        # 打印用户心率分析报告
        print(f"用户 {user_id} 心率分析报告:")
        for key, value in user_analysis.items():
            print(f"{key}: {value}")

        # 创建一个新的DataFrame，只包含特定用户的心率数据
        user_heart_rate_df = model.data[model.data['user_id'] == user_id]

        # 临时替换数据以进行可视化
        model.data = user_heart_rate_df

        # 可视化特定用户的心率分布
        model.visualize_heart_rate_distribution()

In [ ]:
#####            个体-群体对比雷达图 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import plotly.express as px

# 数据解析 
def parse_line(line):
    line = line.strip()
    # 替换错误的小数点分隔符为逗号（如"-556410.36066.57" → "-556410.36066,57"）
    if line.count(".") > 1:
        line = re.sub(r"\.(?=[^.]*\.)", ",", line)  # 只保留最后一个小数点
    return [float(x) for x in line.split(",")]

# 读取文件并处理数据(数据清洗)
data = []
with open("46343_heartrate.txt", "r") as f:
    for line in f:
        if line.strip():  # 跳过空行
            try:
                seconds, bpm = parse_line(line)
                data.append({"seconds": seconds, "bpm": bpm})#将有效数据存储为字典列表，后续转换为DataFrame
            except:
                print(f"解析失败的行: {line}")#尝试解析每行数据，失败时打印警告（避免程序中断）

df = pd.DataFrame(data)
# 转换秒数为相对分钟数（从第一个时间点开始计算）
df["minutes"] = (df["seconds"] - df["seconds"].min()) / 60

# 过滤生理学不合理的心率值（正常范围30-200bpm）
df = df[(df["bpm"] >= 30) & (df["bpm"] <= 200)]

######        一位志愿者bpm和时间的统计散点图 （时间在4085.99）之间
fig = px.scatter(df, x="minutes", y="bpm", 
                 title=f"Heart Rate Dynamics (Volunteer 46343)",
                 labels={"minutes": "Time (minutes)", "bpm": "Heart Rate (bpm)"},
                 trendline="lowess")  # 添加平滑趋势线，避免因数据抖动误判趋势
fig.update_traces(marker=dict(size=4, opacity=0.6))
fig.show()


#####        一位志愿者心率监测图
plt.figure(figsize=(12, 6))
#宽屏比例	展示长时间段趋势时避免图形压缩
plt.plot(df["minutes"], df["bpm"], color="#e74c3c", linewidth=1, alpha=0.8)
plt.title(f"Heart Rate Monitoring (Volunteer 46343)", fontsize=14)
plt.xlabel("Minutes Since First Measurement", fontsize=12)
plt.ylabel("Heart Rate (bpm)", fontsize=12)
plt.grid(alpha=0.3)
#浅灰色网格	辅助读数但不喧宾夺主
plt.tight_layout()
plt.show()

In [ ]:
#####            不同志愿者心率展示悬停图
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

# 准备聚类特征矩阵（保持中文列名）
features = hr_metrics[["平均心率", "静息心率", "心率变异性"]]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

# 执行聚类
kmeans = KMeans(n_clusters=3, random_state=42)
hr_metrics["cluster"] = kmeans.fit_predict(X_scaled)

# 中英文标签映射
label_map = {
    "平均心率": "Average HR",
    "静息心率": "Resting HR",
    "心率变异性": "HR Variability"
}

# 复制数据框避免修改原始数据
plot_data = hr_metrics.copy()
plot_data.rename(columns=label_map, inplace=True)
fig = px.scatter_matrix(plot_data, dimensions=list(label_map.values()),
                       color="cluster", hover_name=plot_data.index)

fig.update_layout(
    # 调整子图间距（单位：像素）
    grid=dict(
        rows=len(label_map), 
        columns=len(label_map),
        xgap=0.1,  # 水平间距
        ygap=0.4   # 垂直间距
    )
)

# 添加中文悬停
fig.update_traces(
    hovertemplate="<br>".join([
        "<b>样本ID</b>: %{hovertext}",
        "<b>平均心率</b>: %{x}",
        "<b>静息心率</b>: %{y}"
    ])
)

fig.show()
fig.write_html("cluster_matrix.html")#保存到文件浏览器

In [ ]:
######            单位志愿者的分析汇总图和单位志愿者综合评估表
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# 导入已有模块
from sleep_evaluate_sys import main as sleep_analysis_main
from activity_analysis import analyze_user_activity
from health_warning_model import HealthWarningModel


class UserProfileAnalyzer:
    def __init__(self, user_id, data_path='cleaned_sleep_data_fixed_time.csv'):
        """
        初始化用户画像分析器

        参数:
        - user_id: 用户ID
        - data_path: 数据文件路径
        """
        self.user_id = user_id
        self.data_path = data_path
        self.data = pd.read_csv(data_path)
        self.user_data = self.data[self.data['user_id'] == user_id]

        # 初始化各个分析模块
        self.sleep_analyzer = None
        self.activity_analyzer = None
        self.health_warning_model = HealthWarningModel(data_path)

    def generate_comprehensive_profile(self):
        """
        生成综合用户画像
        """
        # 睡眠分析
        print("开始睡眠分析...")
        sleep_analysis = self._analyze_sleep()

        # 活动分析
        print("开始活动分析...")
        activity_analysis = self._analyze_activity()

        # 心率分析
        print("开始心率分析...")
        heart_rate_analysis = self._analyze_heart_rate()

        # 生成用户画像
        user_profile = {
            'user_id': self.user_id,
            'sleep_profile': sleep_analysis,
            'activity_profile': activity_analysis,
            'heart_rate_profile': heart_rate_analysis
        }

        return user_profile

    def _analyze_sleep(self):
        """
        分析用户睡眠数据
        """
        # 使用现有的睡眠数据
        sleep_data = self.data[self.data['user_id'] == self.user_id]

        # 计算睡眠相关特征
        sleep_times = sleep_data['time'].diff().dropna()

        # 从现有的睡眠分析函数中提取有用信息
        sleep_profile = {
            'total_records': len(sleep_data),
            'avg_sleep_duration': sleep_times.mean(),
            'sleep_duration_std': sleep_times.std(),
            'sleep_state_distribution': sleep_data['sleep_state'].value_counts(normalize=True).to_dict(),

            # 添加更多睡眠相关指标
            # 睡眠效率
            'sleep_efficiency': sleep_data['sleep_state'].between(1, 5).mean() * 100,
            'avg_sleep_state': sleep_data['sleep_state'].mean(),
            'sleep_state_variance': sleep_data['sleep_state'].var()
        }

        return sleep_profile

    def _analyze_activity(self):
        """
        分析用户活动数据
        """
        # 使用现有的活动分析模块
        user_activity_df, weekday_weekend_analysis = analyze_user_activity(
            self.user_id)

        activity_profile = {
            'activity_level_distribution': user_activity_df['activity_level'].value_counts(normalize=True).to_dict(),
            'weekday_weekend_activity': weekday_weekend_analysis.to_dict()
        }

        return activity_profile

    def _analyze_heart_rate(self):
        """
        分析用户心率数据
        """
        # 使用现有的健康预警模型
        heart_rate_analysis = self.health_warning_model.analyze_user_heart_rate(
            self.user_id)

        return heart_rate_analysis

    def generate_health_recommendations(self, user_profile):
        """
        根据用户画像生成个性化健康建议
        """
        recommendations = []

        # 睡眠建议
        sleep_profile = user_profile['sleep_profile']
        if sleep_profile['sleep_state_distribution'].get(0, 0) > 0.3:  # 清醒时间过长
            recommendations.append("建议优化睡眠环境，减少夜间清醒时间。可以尝试：")
            recommendations.append("- 保持卧室安静、黑暗和凉爽")
            recommendations.append("- 睡前避免使用电子设备")
            recommendations.append("- 建立规律的睡眠作息")

        # 活动建议
        activity_profile = user_profile['activity_profile']
        sedentary_ratio = activity_profile['activity_level_distribution'].get(
            '久坐', 0)
        if sedentary_ratio > 0.5:
            recommendations.append("检测到久坐时间较长，建议：")
            recommendations.append("- 每小时起来走动10-15分钟")
            recommendations.append("- 使用站立式办公桌")
            recommendations.append("- 每天进行至少30分钟中等强度运动")

        # 心率建议
        heart_rate_profile = user_profile['heart_rate_profile']
        if heart_rate_profile.get('warning_level_distribution', {}).get('red', 0) > 0:
            recommendations.append("心率存在高风险，建议：")
            recommendations.append("- 及时就医进行全面体检")
            recommendations.append("- 减少压力，保持良好作息")
            recommendations.append("- 控制饮食，减少咖啡因和酒精摄入")

        # 综合建议
        recommendations.append("\n综合健康建议：")
        recommendations.append("1. 保持规律作息")
        recommendations.append("2. 适度运动")
        recommendations.append("3. 均衡饮食")
        recommendations.append("4. 定期体检")

        return recommendations

    def visualize_user_profile(self, user_profile):
        """
        可视化用户画像
        """
        # 睡眠状态映射
        sleep_state_mapping = {
            0: '清醒',
            1: 'N1入睡期',
            2: 'N2浅睡眠',
            3: 'N3中度睡眠',
            4: 'N4深度睡眠',
            5: 'REM快速眼动期'
        }

        plt.figure(figsize=(15, 10))

        # 睡眠状态分布
        plt.subplot(2, 2, 1)
        sleep_states = user_profile['sleep_profile']['sleep_state_distribution']
        # 映射数字键到文字标签
        mapped_sleep_states = {
            sleep_state_mapping.get(int(k), k): v
            for k, v in sleep_states.items()
        }
        plt.pie(
            list(mapped_sleep_states.values()),
            labels=list(mapped_sleep_states.keys()),
            autopct='%1.1f%%'
        )
        plt.title('睡眠状态分布')

        # 活动强度分布
        plt.subplot(2, 2, 2)
        activity_levels = user_profile['activity_profile']['activity_level_distribution']
        plt.pie(activity_levels.values(), labels=list(
            activity_levels.keys()), autopct='%1.1f%%')
        plt.title('活动强度分布')

        # 心率分布
        plt.subplot(2, 2, 3)
        heart_rate_profile = user_profile['heart_rate_profile']
        plt.hist(self.user_data['heart_rate'], bins=30, edgecolor='black')
        plt.title('心率分布')
        plt.xlabel('心率')
        plt.ylabel('频率')

        # 工作日/周末活动对比
        plt.subplot(2, 2, 4)
        weekday_weekend_activity = user_profile['activity_profile']['weekday_weekend_activity']
        pd.DataFrame(weekday_weekend_activity).plot(kind='bar', ax=plt.gca())
        plt.title('工作日/周末活动对比')
        plt.xlabel('日期类型')
        plt.ylabel('活动强度比例')

        plt.tight_layout()
        plt.savefig(f'user_{self.user_id}_profile.png')
        plt.close()


def save_analysis_to_file(user_profile, health_recommendations, user_id):
    """
    将用户分析结果保存到文件

    参数:
    - user_profile: 用户画像字典
    - health_recommendations: 健康建议列表
    - user_id: 用户ID
    """
    # 睡眠状态映射
    sleep_state_mapping = {
        0: '清醒',
        1: 'N1入睡期',
        2: 'N2浅睡眠',
        3: 'N3中度睡眠',
        4: 'N4深度睡眠',
        5: 'REM快速眼动期'
    }

    filename = f"{user_id}_健康分析汇总.txt"

    with open(filename, 'w', encoding='utf-8') as f:
        # 写入用户ID
        f.write(f"用户ID: {user_id}\n")
        f.write("=" * 50 + "\n\n")

        # 睡眠分析
        f.write("睡眠分析:\n")
        sleep_profile = user_profile['sleep_profile']
        f.write(f"总记录数: {sleep_profile['total_records']}\n")
        f.write(f"平均睡眠时长: {sleep_profile['avg_sleep_duration']:.2f}\n")
        f.write(f"睡眠时长标准差: {sleep_profile['sleep_duration_std']:.2f}\n")
        f.write("睡眠状态分布:\n")
        for state, ratio in sleep_profile['sleep_state_distribution'].items():
            # 使用映射将数字状态转换为文字描述
            state_label = sleep_state_mapping.get(int(state), str(state))
            f.write(f"- {state_label}: {ratio*100:.2f}%\n")
        f.write(f"睡眠效率: {sleep_profile['sleep_efficiency']:.2f}%\n")
        f.write(f"平均睡眠状态: {sleep_profile['avg_sleep_state']:.2f}\n")
        f.write(f"睡眠状态方差: {sleep_profile['sleep_state_variance']:.2f}\n\n")

        # 活动分析
        f.write("活动分析:\n")
        activity_profile = user_profile['activity_profile']
        f.write("活动强度分布:\n")
        for level, ratio in activity_profile['activity_level_distribution'].items():
            f.write(f"- {level}: {ratio*100:.2f}%\n")

        f.write("\n工作日/周末活动对比:\n")
        weekday_weekend_activity = activity_profile['weekday_weekend_activity']
        for day_type, activities in weekday_weekend_activity.items():
            f.write(f"{day_type}:\n")
            for level, ratio in activities.items():
                f.write(f"- {level}: {ratio*100:.2f}%\n")

        # 心率分析
        f.write("\n心率分析:\n")
        heart_rate_profile = user_profile['heart_rate_profile']
        for key, value in heart_rate_profile.items():
            if isinstance(value, (int, float)):
                f.write(f"{key}: {value}\n")
            elif isinstance(value, dict):
                f.write(f"{key}:\n")
                for sub_key, sub_value in value.items():
                    f.write(f"- {sub_key}: {sub_value}\n")

        # 健康建议
        f.write("\n健康建议:\n")
        for recommendation in health_recommendations:
            f.write(f"{recommendation}\n")

    print(f"分析结果已保存到 {filename}")


def main():
    # 选择特定用户进行分析
    user_id = 1066528

    # 初始化用户画像分析器
    analyzer = UserProfileAnalyzer(user_id)

    # 生成用户画像
    user_profile = analyzer.generate_comprehensive_profile()

    # 打印活动强度分析结果
    print("\n用户日常活动强度分析结果：")
    weekday_weekend_activity = user_profile['activity_profile']['weekday_weekend_activity']
    activity_df = pd.DataFrame(weekday_weekend_activity).T
    print(activity_df)

    # 生成健康建议
    health_recommendations = analyzer.generate_health_recommendations(
        user_profile)

    # 可视化用户画像
    analyzer.visualize_user_profile(user_profile)

    # 将分析结果保存到文件
    save_analysis_to_file(user_profile, health_recommendations, user_id)

    # 打印健康建议
    print("\n个性化健康建议：")
    for recommendation in health_recommendations:
        print(recommendation)


if __name__ == "__main__":
    main()